# Activity 0: Environment check

## Part 1 · `nwcst`

Open **Anaconda Prompt** in this `install/` folder and run these lines:

```
conda env create -f environment.yml
conda activate nwcst
python -m ipykernel install --user --name nwcst
```

Select the kernel **`nwcst`** and run the cell below. You should see `LinearRegression OK` and `LSTM OK`. Done.

In [ ]:
# Utilities
import io, os, re, urllib3, requests, datetime, glob, json
import pandas as pd
import numpy as np
from datetime import date
from bs4 import BeautifulSoup
from dateutil.parser import parse
import xml.etree.ElementTree as ET
from io import BytesIO
import zipfile
import eurostat
import pprint as pp
import logging

# Statistics
import matplotlib.pyplot as plt
import pmdarima as pm
import torch
from nowcast_lstm.LSTM import LSTM
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# ── LinearRegression smoke test ───────────────────────────────────────────────
rng = np.random.default_rng(42)
X_train = rng.standard_normal((100, 3))
y_train = X_train @ np.array([1.5, -2.0, 0.8]) + rng.standard_normal(100) * 0.5

X_test = rng.standard_normal((20, 3))
y_test = X_test @ np.array([1.5, -2.0, 0.8]) + rng.standard_normal(20) * 0.5

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)

print(f"LinearRegression OK: R² = {r2:.4f}")
print(f"Coefficients: {model.coef_.round(3)}")


# ── LSTM smoke test ───────────────────────────────────────────────────────────
# Synthetic quarterly data: 40 quarters of training, 4 of test
rng = np.random.default_rng(0)
n_obs = 44
dates = pd.date_range("2010-01-01", periods=n_obs, freq="QS")
x1 = rng.standard_normal(n_obs)
x2 = rng.standard_normal(n_obs)
target = 1.5 * x1 - 2.0 * x2 + rng.standard_normal(n_obs) * 0.3

df = pd.DataFrame({"date": dates, "gdp": target, "x1": x1, "x2": x2})
train_lstm = df.iloc[:40].reset_index(drop=True)
full_lstm  = df.reset_index(drop=True)

lstm_params = {
    "n_timesteps"            : 4,
    "fill_na_func"           : np.nanmean,
    "fill_ragged_edges_func" : np.nanmean,
    "n_models"               : 2,
    "train_episodes"         : 10,
    "batch_size"             : 30,
    "decay"                  : 0.98,
    "n_hidden"               : 8,
    "n_layers"               : 1,
    "dropout"                : 0.0,
    "criterion"              : torch.nn.MSELoss(),
    "optimizer"              : torch.optim.Adam,
    "optimizer_parameters"   : {"lr": 1e-2, "weight_decay": 0.0},
}

lstm_model = LSTM(data=train_lstm, target_variable="gdp", **lstm_params)
lstm_model.train(quiet=True)
preds = lstm_model.predict(full_lstm)

print(f"LSTM OK: predictions shape: {preds.shape}")
print(preds.tail(4).to_string(index=False))

## Part 2 · `geo` (only for Activity 3, nighttime lights)

In the same Anaconda Prompt, run these lines:

```
conda env create -f environment-geo.yml
conda activate geo
python -m ipykernel install --user --name geo
```

Paste your NASA token into `api_key` below (how to get one: `manual_install.txt`), switch the kernel to **`geo`** and run the cell. You should see a map of Trinidad and Tobago and `Geo environment OK`. Done.

In [ ]:
#### Replace API Key with your own (https://ladsweb.modaps.eosdis.nasa.gov/)
api_key = "PASTE_YOUR_LAADS_TOKEN_HERE"


# ── Geo / NTL smoke test: download one h5 file ────────────────────────────────
# A Jupyter kernel skips conda activation, so GDAL can miss its HDF5 plugin
# and read every tile as empty. Point it at this environment's folders first.
import os, sys
_base = os.path.join(sys.prefix, "Library") if os.path.isdir(os.path.join(sys.prefix, "Library")) else sys.prefix
for _var, _sub in (("GDAL_DATA", ("share", "gdal")), ("GDAL_DRIVER_PATH", ("lib", "gdalplugins")), ("PROJ_LIB", ("share", "proj"))):
    if not os.environ.get(_var) and os.path.isdir(os.path.join(_base, *_sub)):
        os.environ[_var] = os.path.join(_base, *_sub)
os.environ["PATH"] = os.path.join(_base, "bin") + os.pathsep + os.environ.get("PATH", "")

import struct, os, sys, time, requests, glob, re, h5py
import numpy as np, numpy.ma as ma
from osgeo import gdal, ogr
import matplotlib.pyplot as plt
import statistics as stat
import pandas as pd
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
import geopandas as gpd
import contextily as cx
import colorcet as cc
import rasterio
from rasterio.mask import mask as rio_mask
from shapely.geometry import mapping

from osgeo import gdal
print(gdal.__version__)
drivers = [gdal.GetDriver(i).ShortName for i in range(gdal.GetDriverCount())]
print("HDF5" in drivers)   # should be True
print("HDF5Image" in drivers)  # should be True

# Target: 2025-10-20, Trinidad and Tobago tile (day 293 of 2025)
year, day, tile = 2025, 293, "h11v07"

out_dir    = Path(os.getcwd()).parent / "hfiles"   # public/hfiles
output_dir = out_dir / "output"
out_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

headers  = {"Authorization": f"Bearer {api_key}"}
base_url = "https://ladsweb.modaps.eosdis.nasa.gov/archive/allData/5200/VNP46A2"

# List files for that day and pick the matching tile
resp      = requests.get(f"{base_url}/{year}/{day:03d}.json", headers=headers)
filenames = resp.json().get("content", [])
matching  = [f["downloadsLink"] for f in filenames if tile in f["downloadsLink"] and f["downloadsLink"].endswith(".h5")]

if not matching:
    print("No matching file found; check API key or date/tile.")
else:
    file_url = matching[0]
    filename = file_url.split("/")[-1]
    out_path = out_dir / filename

    if out_path.exists():
        print(f"Already exists: {filename}")
    else:
        print(f"Downloading: {filename}")
        r = requests.get(file_url, headers=headers, stream=True)
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Downloaded: {filename}")

    # Verify with h5py
    with h5py.File(out_path, "r") as hf:
        print(f"h5py OK: top-level keys: {list(hf.keys())}")

    # Verify gdal
    hdf = gdal.Open(str(out_path), gdal.GA_ReadOnly)
    sds = hdf.GetSubDatasets()
    print(f"gdal OK: {len(sds)} subdatasets found")
    hdf = None

    # ── Map the downloaded file ────────────────────────────────────────────────
    gdf     = gpd.read_file("https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_TTO_shp.zip")
    gdf_tto = gdf[gdf.geometry.notnull()].dissolve().to_crs("EPSG:4326")

    hdflayer    = gdal.Open(str(out_path), gdal.GA_ReadOnly)
    subhdflayer = hdflayer.GetSubDatasets()[2][0]   # layer 2 = DNB_At_Sensor_Radiance
    rlayer      = gdal.Open(subhdflayer, gdal.GA_ReadOnly)

    meta        = rlayer.GetMetadata_Dict()
    H, V        = int(meta["HorizontalTileNumber"]), int(meta["VerticalTileNumber"])
    west, north = (10 * H) - 180, 90 - (10 * V)
    east, south = west + 10, north - 10

    tmp  = str(output_dir / "_test_tmp.tif")
    opts = gdal.TranslateOptions(gdal.ParseCommandLine(
        f"-a_srs EPSG:4326 -a_ullr {west} {north} {east} {south}"
    ))
    gdal.Translate(tmp, rlayer, options=opts)
    rlayer = None

    shapes = [mapping(geom) for geom in gdf_tto.geometry]
    with rasterio.open(tmp) as src:
        out_image, out_transform = rio_mask(src, shapes, crop=True, nodata=np.nan, filled=True)
        nrows, ncols = out_image.shape[1], out_image.shape[2]
        lons = out_transform.c + (np.arange(ncols) + 0.5) * out_transform.a
        lats = out_transform.f + (np.arange(nrows) + 0.5) * out_transform.e

    try: os.remove(tmp)
    except: pass

    data = out_image[0].astype(float)
    data[data >= 65535] = np.nan
    data[data < 0]      = np.nan

    LON, LAT   = np.meshgrid(lons, lats)
    valid      = data[~np.isnan(data)]
    vmin, vmax = np.nanpercentile(valid, 2), np.nanpercentile(valid, 98)
    date_label = datetime.strptime(f"{year}-{day:03d}", "%Y-%j").strftime("%b %d, %Y")

    fig, ax = plt.subplots(figsize=(10, 6))
    pcm = ax.pcolormesh(LON, LAT, data, cmap=cc.cm.bmy, vmin=vmin, vmax=vmax, shading="auto")
    cx.add_basemap(ax, source=cx.providers.CartoDB.DarkMatter, crs="EPSG:4326")
    fig.colorbar(pcm, ax=ax, orientation="vertical", label="NTL Radiance (nW/cm²/sr)")
    ax.set_title(f"Nighttime Lights, Trinidad and Tobago ({date_label})", fontsize=13, weight="bold")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    fig.text(0.01, 0.01, "Source: NASA Black Marble VNP46A2", fontsize=9)
    plt.tight_layout()
    plt.show()

    # ── Cleanup ───────────────────────────────────────────────────────────────
    #out_path.unlink()
    #print(f"Deleted: {filename}")
    print("Geo environment OK")

## Part 3 · Set up the R paths in VS Code

This cell writes the `r.rpath` / `r.rterm` keys to the VS Code `settings.json`, so the R extension finds your R installation without configuring it by hand.

**How to use it:** run the cell below once (it only uses Python, the kernel of this notebook). It finds R even if it is not on the PATH, keeps whatever you already have in `settings.json` (it leaves a `.bak` backup), and works on Windows, macOS and Linux. When it finishes, **close VS Code completely and reopen it**.

**Alternative:** if R is already on the system PATH, you do not need this; the extension finds it on its own.

In [ ]:
# Set the R paths in the VS Code settings.json
# Run this cell ONCE. It only needs Python (the kernel of this notebook).
# When it finishes, CLOSE VS Code completely and reopen it.

import json, os, re, glob, platform
from pathlib import Path

system = platform.system()  # 'Windows', 'Darwin', 'Linux'

# ---------- 1) Find the R executables ----------
def ver_key(p):
    m = re.search(r"R-([\d.]+)", str(p))
    return tuple(int(x) for x in m.group(1).split(".")) if m else (0,)

def find_r():
    """Returns (rpath, rterm) with forward slashes, or (None, None)."""
    if system == "Windows":
        bases = [r"C:\Program Files\R", r"C:\Program Files (x86)\R",
                 os.path.expandvars(r"%LOCALAPPDATA%\Programs\R")]
        bindirs = []
        for base in bases:
            bindirs += [Path(g).parent for g in glob.glob(os.path.join(base, "R-*", "bin", "x64", "R.exe"))]
            bindirs += [Path(g).parent for g in glob.glob(os.path.join(base, "R-*", "bin", "R.exe"))]
        for b in sorted(set(bindirs), key=ver_key, reverse=True):
            if (b / "R.exe").exists() and (b / "Rterm.exe").exists():
                return str(b / "R.exe").replace("\\", "/"), str(b / "Rterm.exe").replace("\\", "/")
        return None, None
    else:
        import shutil
        rbin = shutil.which("R")
        return (rbin, rbin) if rbin else (None, None)

rpath, rterm = find_r()
if not rpath:
    raise SystemExit(
        "Could not find R automatically. Open R (RGui) and run R.home('bin') "
        "to see the path, then add these keys to settings.json by hand:\n"
        '  "r.rpath.windows": "<ruta>/R.exe",\n  "r.rterm.windows": "<ruta>/Rterm.exe"')
print("R found:")
print("  rpath:", rpath)
print("  rterm:", rterm)

# ---------- 2) Find the user's settings.json ----------
if system == "Windows":
    settings = Path(os.environ["APPDATA"]) / "Code" / "User" / "settings.json"
elif system == "Darwin":
    settings = Path.home() / "Library" / "Application Support" / "Code" / "User" / "settings.json"
else:
    settings = Path.home() / ".config" / "Code" / "User" / "settings.json"
settings.parent.mkdir(parents=True, exist_ok=True)

# ---------- 3) Read settings.json, allowing comments (JSONC) ----------
def strip_jsonc(text):
    text = re.sub(r"/\*.*?\*/", "", text, flags=re.S)   # /* */ blocks
    out = []
    for line in text.splitlines():                       # // outside strings
        in_str = esc = False; res = ""; i = 0
        while i < len(line):
            c = line[i]
            if esc: res += c; esc = False
            elif c == "\\" and in_str: res += c; esc = True
            elif c == '"': in_str = not in_str; res += c
            elif c == "/" and i + 1 < len(line) and line[i+1] == "/" and not in_str: break
            else: res += c
            i += 1
        out.append(res)
    return re.sub(r",(\s*[}\]])", r"\1", "\n".join(out))  # trailing commas

data = {}
if settings.exists() and settings.stat().st_size > 0:
    raw = settings.read_text(encoding="utf-8")
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        try:
            data = json.loads(strip_jsonc(raw))
        except json.JSONDecodeError:
            raise SystemExit(
                f"Could not parse {settings}. Add these keys by hand:\n"
                f'  "r.rpath.{ "windows" if system=="Windows" else "mac" if system=="Darwin" else "linux" }": "{rpath}",\n'
                f'  "r.rterm.{ "windows" if system=="Windows" else "mac" if system=="Darwin" else "linux" }": "{rterm}"')

# ---------- 4) Set the keys for the right platform ----------
suf = "windows" if system == "Windows" else "mac" if system == "Darwin" else "linux"
data[f"r.rpath.{suf}"] = rpath
data[f"r.rterm.{suf}"] = rterm

# ---------- 5) Backup and write ----------
if settings.exists():
    settings.with_suffix(".json.bak").write_text(settings.read_text(encoding="utf-8"), encoding="utf-8")
settings.write_text(json.dumps(data, indent=4, ensure_ascii=False) + "\n", encoding="utf-8")

print("\nsettings.json updated:", settings)
print("Backup:", settings.with_suffix(".json.bak"))
print("\nDone. CLOSE VS Code completely and reopen it so it picks up the paths.")